In [4]:
# plain_weave_from_tshirts.py
# pip install opencv-python pillow numpy

import cv2, numpy as np
from PIL import Image
from pathlib import Path

# ----- 입력 이미지 경로(임의 변경 가능)
IMGS = [
    "images.jpg",        # 스누피
#    "tshirt.jpg",        # FLORIDA 네이비
    "다운로드.jpg",       # 블랙(무지)
#    "images (1).jpg",    # Vans 뒷프린트
]

OUTDIR = Path("./out_plainweave"); OUTDIR.mkdir(parents=True, exist_ok=True)

# ===== 공통 유틸 =====
def imread_any(p):
    p = str(p)
    img = cv2.imdecode(np.fromfile(p, dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None: img = cv2.imread(p, cv2.IMREAD_COLOR)
    if img is None: raise RuntimeError(f"Cannot load {p}")
    return img

def segment_shirt(img):
    h,w = img.shape[:2]; pad = int(min(h,w)*0.06)
    rect = (pad,pad,w-2*pad,h-2*pad)
    mask = np.zeros((h,w), np.uint8); bg=np.zeros((1,65),np.float64); fg=np.zeros((1,65),np.float64)
    try:
        cv2.grabCut(img, mask, rect, bg, fg, 5, cv2.GC_INIT_WITH_RECT)
        mask = ((mask==cv2.GC_FGD)|(mask==cv2.GC_PR_FGD)).astype(np.uint8)
    except:
        small = cv2.resize(img, (min(320,w), int(h*min(320,w)/w)))
        Z = small.reshape((-1,3)).astype(np.float32)
        _, lab, _ = cv2.kmeans(Z, 2, None, (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0), 10, cv2.KMEANS_PP_CENTERS)
        lab = lab.reshape(small.shape[:2]); bg = np.argmax(np.bincount(lab.ravel()))
        mask = cv2.resize((lab!=bg).astype(np.uint8), (w,h), cv2.INTER_NEAREST)
    k = max(3, (min(h,w)//150)|1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k,k))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, 2)
    mask = cv2.medianBlur((mask*255).astype(np.uint8), 5)
    return (mask>0).astype(np.uint8)

def pick_spot(img, mask, rel=0.42, mode="max"):
    """mode='max' → 로고/문양 많은 영역(에지 최댓값), mode='min' → 무지에 가까운 영역"""
    h,w = mask.shape; size = int(rel*min(h,w)); size = max(80, min(size, min(h,w)-10))
    dist = cv2.distanceTransform(mask*255, cv2.DIST_L2, 3)
    safe = (dist > max(6, int(size*0.06))).astype(np.uint8)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Sobel(gray, cv2.CV_32F, 1, 1, ksize=3)
    edges = cv2.GaussianBlur(np.abs(edges), (0,0), 1.0); edges *= mask.astype(np.float32)
    ii = cv2.integral(edges)
    best = None; bestxy = (w//2-size//2, h//2-size//2)
    step = max(4, size//10)
    for y in range(0, h-size, step):
        for x in range(0, w-size, step):
            if not safe[y+size//2, x+size//2]: continue
            s = ii[y+size,x+size] - ii[y,x+size] - ii[y+size,x] + ii[y,x]
            if best is None or (mode=="max" and s>best) or (mode=="min" and s<best):
                best, bestxy = s, (x,y)
    x,y = bestxy
    patch = img[y:y+size, x:x+size].copy()
    outside = (mask[y:y+size, x:x+size]==0).astype(np.uint8)*255
    if outside.any(): patch = cv2.inpaint(patch, outside, 3, cv2.INPAINT_TELEA)
    patch = cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)
    # 디테일 유지를 위해 2x 업샘플
    patch = cv2.resize(patch, (patch.shape[1]*2, patch.shape[0]*2), interpolation=cv2.INTER_LANCZOS4)
    return patch, (x,y,size,size)

def build_weave(patches, H=2048, W=2048, warp_w=12, weft_h=12, seed=777):
    rng = np.random.default_rng(seed)
    tiles = patches
    warp = np.zeros((H,W,3), np.uint8); weft = np.zeros((H,W,3), np.uint8)
    cx=[0]*len(tiles)
    for x0 in range(0,W,warp_w):
        x1=min(W,x0+warp_w); idx=(x0//warp_w)%len(tiles); t=tiles[idx]
        if cx[idx]+(x1-x0)>=t.shape[1]: cx[idx]=int(rng.integers(0, max(1,t.shape[1]//3)))
        col=t[:H, cx[idx]:cx[idx]+(x1-x0)]
        if col.shape[0]<H: col=np.vstack([col]*(H//col.shape[0]+1))[:H]
        warp[:,x0:x1]=col[:H,:x1-x0]; cx[idx]+=(x1-x0)
    cy=[0]*len(tiles)
    for y0 in range(0,H,weft_h):
        y1=min(H,y0+weft_h); idx=((y0//weft_h)+1)%len(tiles); t=tiles[idx]
        if cy[idx]+(y1-y0)>=t.shape[0]: cy[idx]=int(rng.integers(0, max(1,t.shape[0]//3)))
        row=t[cy[idx]:cy[idx]+(y1-y0), :W]
        if row.shape[1]<W: row=np.hstack([row]*(W//row.shape[1]+1))[:,:W]
        weft[y0:y1]=row[:(y1-y0),:W]; cy[idx]+=(y1-y0)

    xx=np.arange(W)[None,:]; yy=np.arange(H)[:,None]
    warp_top=((xx//warp_w + yy//weft_h) % 2 == 0)
    warp_pos=(xx%warp_w)/max(1,warp_w-1); weft_pos=(yy%weft_h)/max(1,weft_h-1)
    warp_sh=(1+0.10*np.cos((warp_pos-0.5)*np.pi*2))[...,None]
    weft_sh=(1+0.10*np.cos((weft_pos-0.5)*np.pi*2))[...,None]
    TOP,UNDER=1.07,0.84
    warp_top_img=np.clip(warp.astype(np.float32)*warp_sh*TOP,0,255)
    weft_top_img=np.clip(weft.astype(np.float32)*weft_sh*TOP,0,255)
    warp_under_img=np.clip(warp.astype(np.float32)*warp_sh*UNDER,0,255)
    weft_under_img=np.clip(weft.astype(np.float32)*weft_sh*UNDER,0,255)
    out=np.where(warp_top[...,None], warp_top_img, weft_top_img)
    warp_edge=(np.abs((xx%warp_w)-warp_w/2)>(warp_w*0.35)).astype(np.float32)[...,None]
    weft_edge=(np.abs((yy%weft_h)-weft_h/2)>(weft_h*0.35)).astype(np.float32)[...,None]
    under=np.where(warp_top[...,None], weft_under_img, warp_under_img)
    edge=np.where(warp_top[...,None], warp_edge, weft_edge)
    out=np.clip(out*(1-0.17*edge)+under*0.17*edge,0,255).astype(np.uint8)

    warp_h=(0.5+0.5*np.cos((warp_pos-0.5)*np.pi*2))
    weft_hv=(0.5+0.5*np.cos((weft_pos-0.5)*np.pi*2))
    height=np.where(warp_top, np.maximum(warp_h,weft_hv), np.maximum(weft_hv,warp_h)).astype(np.float32)
    height=(height-height.min())/(height.max()-height.min()+1e-8)
    return out, height

def make_tileable(img):
    a=img.copy(); H,W=a.shape[:2]
    a=np.roll(a,H//2,axis=0); a=np.roll(a,W//2,axis=1)
    blur=cv2.GaussianBlur(a,(0,0),3)
    yy=np.arange(H)[:,None]; xx=np.arange(W)[None,:]
    mask_y=np.clip(1-np.abs(yy-H//2)/(H*0.12),0,1)
    mask_x=np.clip(1-np.abs(xx-W//2)/(W*0.12),0,1)
    out = blur*(mask_y*mask_x)[...,None] + a*(1-(mask_y*mask_x)[...,None])
    out=np.roll(out,-H//2,axis=0); out=np.roll(out,-W//2,axis=1)
    return np.clip(out,0,255).astype(np.uint8)

def height_to_normal(h, strength=4.0):
    dx=cv2.Sobel(h,cv2.CV_32F,1,0,ksize=3); dy=cv2.Sobel(h,cv2.CV_32F,0,1,ksize=3)
    nx=-dx*strength; ny=dy*strength; nz=np.ones_like(h)
    n=np.stack([nx,ny,nz],axis=2); n/= (np.linalg.norm(n,axis=2,keepdims=True)+1e-8)
    return ((n+1)*0.5*255).astype(np.uint8)

def run(mode_name, mode_sel, seed):
    imgs=[imread_any(p) for p in IMGS]
    masks=[segment_shirt(im) for im in imgs]
    patches=[]; boxes=[]
    for im, m in zip(imgs, masks):
        patch, box = pick_spot(im, m, rel=0.42, mode=mode_sel)
        patches.append(patch); boxes.append(box)

    albedo, height = build_weave(patches, H=2048, W=2048, warp_w=50, weft_h=50, seed=seed)
    tile = make_tileable(albedo)
    normal = height_to_normal(height.astype(np.float32), strength=4.0)
    height_u8 = (np.clip(height,0,1)*255).astype(np.uint8)
    rough = np.full((albedo.shape[0], albedo.shape[1]), 180, dtype=np.uint8)  # ~0.7

    def save(name, arr): Image.fromarray(arr).save(OUTDIR/name); return str(OUTDIR/name)

    files = {
        "albedo": save(f"{mode_name}_albedo.png", albedo),
        "albedo_tileable": save(f"{mode_name}_albedo_tileable.png", tile),
        "height": save(f"{mode_name}_height.png", height_u8),
        "normal": save(f"{mode_name}_normal.png", normal),
        "roughness": save(f"{mode_name}_roughness.png", rough),
        "zoom": save(f"{mode_name}_zoom.png", albedo[896:1152,896:1152]),
    }
    # 스폿 프리뷰/패치
    for i,(im,m,b,p) in enumerate(zip(imgs, masks, boxes, patches), start=1):
        x,y,w,h = b
        vis = im.copy(); vis[m==0]=(vis[m==0]*0.6).astype(np.uint8)
        cv2.rectangle(vis,(x,y),(x+w,y+h),(0,255,0),max(2,vis.shape[1]//300))
        cv2.imwrite(str(OUTDIR/f"{mode_name}_spot_preview_{i}.png"), vis)
        Image.fromarray(p).save(OUTDIR/f"{mode_name}_spot_patch_{i}.png")
    return files

if __name__ == "__main__":
    print(">> logo-heavy (로고 많이)")
    heavy = run("logo_heavy", "max", 777)
    print(heavy)
    print(">> logo-light (로고 적게)")
    light = run("logo_light", "min", 779)
    print(light)
    print(f"All done. See folder: {OUTDIR.resolve()}")


>> logo-heavy (로고 많이)
{'albedo': 'out_plainweave\\logo_heavy_albedo.png', 'albedo_tileable': 'out_plainweave\\logo_heavy_albedo_tileable.png', 'height': 'out_plainweave\\logo_heavy_height.png', 'normal': 'out_plainweave\\logo_heavy_normal.png', 'roughness': 'out_plainweave\\logo_heavy_roughness.png', 'zoom': 'out_plainweave\\logo_heavy_zoom.png'}
>> logo-light (로고 적게)
{'albedo': 'out_plainweave\\logo_light_albedo.png', 'albedo_tileable': 'out_plainweave\\logo_light_albedo_tileable.png', 'height': 'out_plainweave\\logo_light_height.png', 'normal': 'out_plainweave\\logo_light_normal.png', 'roughness': 'out_plainweave\\logo_light_roughness.png', 'zoom': 'out_plainweave\\logo_light_zoom.png'}
All done. See folder: C:\Users\_idal\PycharmProjects\idal\Cloth\out_plainweave


In [16]:
# thread_weave_swatch.py
# Pillow + NumPy only

from PIL import Image, ImageFilter
import numpy as np
from pathlib import Path

# ========== 설정 ==========
# 입력 파일 (현재 폴더에 있다고 가정; 경로 자유롭게 바꿔도 됨)
PATH_TEE_A = Path("다운로드.jpg")        # 경사(warp)에 쓸 티셔츠
PATH_TEE_B = Path("images (1).jpg")     # 위사(weft)에 쓸 티셔츠

OUT_DIR = Path("./out_thread_weave"); OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_SIZE = 2048         # 2048 권장 (4096도 가능)
YARNS    = 32           # 실 개수: 16(굵직), 32(보통), 40(조밀)
N_SWATCH = 3            # 랜덤 스와치 몇 장 생성할지
RAND_SEED_BASE = 1234   # 재현 가능한 결과 원하면 고정, 매번 다르게면 None

# 랜덤 크롭 크기 범위(정사각형 한 변 비율, 최소~최대)
RANGE_WARP = (0.50, 0.65)
RANGE_WEFT = (0.45, 0.60)

# ========== 리샘플링 호환 ==========
try:
    RES_LANCZOS  = Image.Resampling.LANCZOS
    RES_BILINEAR = Image.Resampling.BILINEAR
    RES_NEAREST  = Image.Resampling.NEAREST
except AttributeError:
    RES_LANCZOS  = Image.LANCZOS
    RES_BILINEAR = Image.BILINEAR
    RES_NEAREST  = Image.NEAREST

# ========== 유틸 ==========
def load_rgb(p: Path) -> Image.Image:
    return Image.open(p).convert("RGB")

def to_np(img: Image.Image) -> np.ndarray:
    return np.asarray(img, np.float32) / 255.0

def to_img(arr: np.ndarray) -> Image.Image:
    return Image.fromarray(np.clip(arr * 255, 0, 255).astype(np.uint8))

def kmeans_simple(X: np.ndarray, k=3, iters=12, seed=0):
    """아주 단순한 K-Means (RGB 공간)"""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), k, replace=False)
    C = X[idx].copy()
    for _ in range(iters):
        D = ((X[:, None, :] - C[None, :, :]) ** 2).sum(2)
        L = D.argmin(1)
        C_new = []
        for j in range(k):
            sel = X[L == j]
            C_new.append(sel.mean(0) if len(sel) else C[j])
        C_new = np.stack(C_new, 0)
        if np.allclose(C_new, C): break
        C = C_new
    return C, L

def tshirt_mask(img: Image.Image, k=3, down=520, border=10) -> Image.Image:
    """티셔츠만 분할한 이진 마스크(L, 0/255)"""
    w, h = img.size
    if w >= h: small = img.resize((down, int(h * down / w)), RES_LANCZOS)
    else:      small = img.resize((int(w * down / h), down), RES_LANCZOS)
    sw, sh = small.size
    X = to_np(small).reshape(-1, 3)
    _, L = kmeans_simple(X, k=k, iters=15, seed=1)
    lab = L.reshape(sh, sw)

    # 테두리에서 가장 많이 보이는 라벨 = 배경
    border_m = np.zeros((sh, sw), bool)
    border_m[:border, :] = border_m[-border:, :] = True
    border_m[:, :border] = True; border_m[:, -border:] = True
    bg_counts = [np.sum((lab == j) & border_m) for j in range(k)]
    bg = int(np.argmax(bg_counts))

    m_small = (lab != bg).astype(np.uint8) * 255
    m_small = Image.fromarray(m_small)
    # 작은 구멍 메움 + 가장자리 정리
    m_small = m_small.filter(ImageFilter.MedianFilter(3))
    m_small = m_small.filter(ImageFilter.MaxFilter(5)).filter(ImageFilter.MinFilter(5))
    return m_small.resize((w, h), RES_NEAREST)

def masked_random_square_crop(img: Image.Image, mask: Image.Image,
                              size_ratio_range=(0.45, 0.65),
                              down=288, cover=1.0, step_div=8, seed=None) -> Image.Image:
    """
    마스크 내부(cover=1.0이면 100%)만 포함하는 정사각형 후보 중 랜덤 선택.
    """
    rng = np.random.default_rng(seed)
    w, h = img.size
    if w >= h:
        sw, sh = down, int(h * down / w)
    else:
        sw, sh = int(w * down / h), down
    msmall = mask.resize((sw, sh), RES_NEAREST)
    M = (np.asarray(msmall, np.uint8) > 127).astype(np.float32)
    IM = M.cumsum(0).cumsum(1)

    def isum(I, x0, y0, s):
        x1, y1 = x0 + s - 1, y0 + s - 1
        a = I[y1, x1]
        b = I[y0 - 1, x1] if y0 > 0 else 0.0
        c = I[y1, x0 - 1] if x0 > 0 else 0.0
        d = I[y0 - 1, x0 - 1] if (x0 > 0 and y0 > 0) else 0.0
        return a - b - c + d

    ratios = np.linspace(size_ratio_range[1], size_ratio_range[0], num=8)
    ratios = ratios[rng.permutation(len(ratios))]
    for r in ratios:
        side = int(min(sw, sh) * r)
        if side < 8:
            continue
        step = max(1, side // step_div)
        req = cover * side * side
        cand = []
        for y0 in range(0, sh - side, step):
            for x0 in range(0, sw - side, step):
                if isum(IM, x0, y0, side) >= req:
                    cand.append((x0, y0, side))
        if cand:
            x0s, y0s, s = cand[rng.integers(len(cand))]
            sx, sy = w / sw, h / sh
            side_o = int(s * min(sx, sy))
            x0o, y0o = int(x0s * sx), int(y0s * sy)
            x0o = max(0, min(w - side_o, x0o))
            y0o = max(0, min(h - side_o, y0o))
            return img.crop((x0o, y0o, x0o + side_o, y0o + side_o))

    # 폴백: 마스크 바운딩박스 중앙 정사각형
    ys, xs = np.where(M > 0)
    y0m, y1m = int(ys.min()), int(ys.max())
    x0m, x1m = int(xs.min()), int(xs.max())
    cw, ch = x1m - x0m + 1, y1m - y0m + 1
    side = int(min(cw, ch) * 0.9)
    bx = x0m + (cw - side) // 2
    by = y0m + (ch - side) // 2
    sx, sy = w / sw, h / sh
    side_o = int(side * min(sx, sy))
    x0o, y0o = int(bx * sx), int(by * sy)
    x0o = max(0, min(w - side_o, x0o))
    y0o = max(0, min(h - side_o, y0o))
    return img.crop((x0o, y0o, x0o + side_o, y0o + side_o))

def lowfreq_noise(shape, seed=0):
    rng = np.random.default_rng(seed); h, w = shape[:2]
    small = rng.random((max(8, h // 64), max(8, w // 64))).astype(np.float32)
    n = Image.fromarray((small * 255).astype(np.uint8)).resize((w, h), RES_BILINEAR)
    n = np.asarray(n, np.float32) / 255.0
    return 0.92 + 0.16 * (n - n.min()) / (n.max() - n.min() + 1e-6)

def thread_weave(warp_crop: Image.Image, weft_crop: Image.Image,
                 out_size=2048, yarns=32, twist_freq=0.7, seed=7) -> Image.Image:
    """실 단위로 plain-weave 합성"""
    rng = np.random.default_rng(seed)
    W = H = out_size
    pitch = W / yarns
    centers_x = (np.arange(yarns) + 0.5) * pitch
    centers_y = (np.arange(yarns) + 0.5) * pitch

    sigma_base = pitch * 0.33
    sigma_warp = (sigma_base * rng.uniform(0.9, 1.1, yarns)).astype(np.float32)
    sigma_weft = (sigma_base * rng.uniform(0.9, 1.1, yarns)).astype(np.float32)
    gain_warp  = rng.uniform(0.96, 1.04, yarns).astype(np.float32)
    gain_weft  = rng.uniform(0.96, 1.04, yarns).astype(np.float32)
    phase_warp = rng.uniform(0, 2*np.pi, yarns).astype(np.float32)
    phase_weft = rng.uniform(0, 2*np.pi, yarns).astype(np.float32)

    warp_src  = warp_crop.resize((yarns, H), RES_LANCZOS)
    warp_cols = to_np(warp_src)                        # H x yarns x 3
    weft_src  = weft_crop.rotate(90, expand=True).resize((W, yarns), RES_LANCZOS)
    weft_rows = to_np(weft_src)                        # yarns x W x 3

    yy, xx = np.meshgrid(np.arange(H, dtype=np.float32),
                         np.arange(W, dtype=np.float32), indexing='ij')
    warp_idx = np.floor(xx / pitch).astype(np.int32).clip(0, yarns - 1)
    weft_idx = np.floor(yy / pitch).astype(np.int32).clip(0, yarns - 1)

    dx = xx - centers_x[warp_idx]
    dy = yy - centers_y[weft_idx]

    sig_w = sigma_warp[warp_idx]
    sig_f = sigma_weft[weft_idx]
    g_w   = gain_warp[warp_idx][..., None]
    g_f   = gain_weft[weft_idx][..., None]
    ph_w  = phase_warp[warp_idx]
    ph_f  = phase_weft[weft_idx]

    warp_body = np.exp(-0.5 * (dx / sig_w) ** 2)
    weft_body = np.exp(-0.5 * (dy / sig_f) ** 2)
    warp_shade = 0.72 + 0.25 * warp_body
    weft_shade = 0.72 + 0.25 * weft_body
    warp_twist = 1.0 + 0.10 * np.cos(2*np.pi*twist_freq * (yy / pitch) + ph_w) * np.exp(-0.5 * (dx / (sig_w * 0.8)) ** 2)
    weft_twist = 1.0 + 0.10 * np.cos(2*np.pi*twist_freq * (xx / pitch) + ph_f) * np.exp(-0.5 * (dy / (sig_f * 0.8)) ** 2)

    warp_rgb = warp_cols[np.arange(H)[:, None], warp_idx] * g_w
    weft_rgb = weft_rows[weft_idx, np.arange(W)[None, :]] * g_f
    warp_rgb *= warp_shade[..., None] * warp_twist[..., None]
    weft_rgb *= weft_shade[..., None] * weft_twist[..., None]

    over_warp = ((warp_idx + weft_idx) % 2 == 0)        # plain 교차
    overlap = warp_body * weft_body
    shadow = 0.65 + 0.35 * (1 - overlap / (overlap.max() + 1e-6))
    vis_under_if_warp_top = (1.0 - warp_body * 0.94) * shadow
    vis_under_if_weft_top = (1.0 - weft_body * 0.94) * shadow

    final = np.zeros((H, W, 3), np.float32)
    m = over_warp
    final[m]  = warp_rgb[m] + weft_rgb[m] * vis_under_if_warp_top[m][..., None]
    final[~m] = weft_rgb[~m] + warp_rgb[~m] * vis_under_if_weft_top[~m][..., None]

    n = lowfreq_noise(final.shape, seed=seed)[..., None]
    final = np.clip(final * n, 0, 1)
    return to_img(final)

# ========== 실행 ==========
if __name__ == "__main__":
    teeA = load_rgb(PATH_TEE_A)
    teeB = load_rgb(PATH_TEE_B)

    # 1) 마스크 생성(배경 제거)
    maskA = tshirt_mask(teeA, k=3, down=520, border=10)
    maskB = tshirt_mask(teeB, k=3, down=520, border=10)
    maskA.save(OUT_DIR / "maskA.png")
    maskB.save(OUT_DIR / "maskB.png")

    # 2) 랜덤 스와치 여러 장 생성 (항상 마스크 내부 100%)
    for i in range(N_SWATCH):
        seed_crop = None if RAND_SEED_BASE is None else (RAND_SEED_BASE + i)
        warp_crop = masked_random_square_crop(teeA, maskA, RANGE_WARP, cover=1.0, seed=seed_crop)
        weft_crop = masked_random_square_crop(teeB, maskB, RANGE_WEFT, cover=1.0, seed=seed_crop)

        warp_crop.save(OUT_DIR / f"warp_crop_{i:02d}.png")
        weft_crop.save(OUT_DIR / f"weft_crop_{i:02d}.png")

        img = thread_weave(warp_crop, weft_crop,
                           out_size=OUT_SIZE, yarns=YARNS,
                           twist_freq=0.7,
                           seed=None if RAND_SEED_BASE is None else (RAND_SEED_BASE + 100 + i))
        out_path = OUT_DIR / f"swatch_threaded_{OUT_SIZE}px_{YARNS}y_{i:02d}.png"
        img.save(out_path)
        print("Saved:", out_path)


Saved: out_thread_weave\swatch_threaded_2048px_32y_00.png
Saved: out_thread_weave\swatch_threaded_2048px_32y_01.png
Saved: out_thread_weave\swatch_threaded_2048px_32y_02.png
